# Lab 4 — The new arrivals join the residents’ meeting

The Metric View family got useful reinforcements: per-measure `FILTER`, window `offset`, and relationships between Metric Views. This lab exercises all three against the same TPC-H evidence. The first two change query semantics; the relationship is metadata for AI/BI consumers, not a magical query-time join.

Run this notebook on a SQL warehouse or DBR 19+ compute. The relationship syntax is Beta and requires Unity Catalog permissions on both Metric Views. Dated window offsets and YAML 1.1 modifiers require a supported runtime; check the live feature matrix before treating a passing lab as a production contract.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.metric_views_lab;

## 1. Prove the filtered measure is not a model-wide filter

Lab 1 added `fulfilled_revenue` with `SUM(...) FILTER (WHERE ...)`. Grouping the same view by status keeps all statuses visible while the filtered measure contributes only to `F`. Compare it with the direct control; if they disagree, the meeting has found a real bug rather than a personality clash.

In [ ]:
%sql
SELECT order_status,
       MEASURE(total_revenue) AS all_status_revenue,
       MEASURE(fulfilled_revenue) AS fulfilled_revenue
FROM workspace.metric_views_lab.orders_metric_view
GROUP BY order_status
ORDER BY order_status;

In [ ]:
%sql
SELECT SUM(o_totalprice) FILTER (WHERE o_orderstatus = 'F') AS direct_fulfilled_revenue,
       SUM(o_totalprice) AS direct_all_status_revenue
FROM samples.tpch.orders
WHERE o_orderdate >= DATE'1995-01-01';

## 2. Slide the window twelve months backwards

A dated `offset` lets a measure look at the corresponding period instead of forcing the dashboard to invent a self-join. `-12 month` is preferable to `-365 day` for month-grain year-over-year work because calendar arithmetic respects leap years and variable-length months. `current` selects the shifted anchor; the query still asks for the current month.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metric_views_lab.monthly_revenue_offset_metric_view
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Monthly revenue with a twelve-month comparison window."
source: samples.tpch.orders
filter: o_orderdate >= DATE'1995-01-01'
fields:
  - name: order_month
    expr: DATE_TRUNC('MONTH', o_orderdate)
measures:
  - name: current_revenue
    expr: SUM(o_totalprice)
  - name: prior_year_revenue
    expr: SUM(o_totalprice)
    window:
      - order: order_month
        range: current
        offset: -12 month
$$;

In [ ]:
%sql
SELECT order_month,
       MEASURE(current_revenue) AS current_revenue,
       MEASURE(prior_year_revenue) AS prior_year_revenue
FROM workspace.metric_views_lab.monthly_revenue_offset_metric_view
GROUP BY order_month
ORDER BY order_month;

In [ ]:
%sql
WITH monthly AS (
  SELECT DATE_TRUNC('MONTH', o_orderdate) AS order_month, SUM(o_totalprice) AS revenue
  FROM samples.tpch.orders
  WHERE o_orderdate >= DATE'1995-01-01'
  GROUP BY DATE_TRUNC('MONTH', o_orderdate)
)
SELECT cur.order_month, cur.revenue AS current_revenue, prior.revenue AS direct_prior_year_revenue
FROM monthly cur
LEFT JOIN monthly prior ON prior.order_month = ADD_MONTHS(cur.order_month, -12)
ORDER BY cur.order_month;

## 3. Put two independently governed views on the noticeboard

A relationship records how two Metric Views fit together so AI assistants and BI tools can choose a join when an analysis needs it. It does not inject a fixed join into every query, enforce uniqueness, or optimize the plan. We deliberately declare `MANY TO ONE` only after checking the customer key in Lab 1.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metric_views_lab.customers_metric_view
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Customer attributes and one-row-per-customer count."
source: samples.tpch.customer
fields:
  - name: customer_key
    expr: c_custkey
  - name: market_segment
    expr: c_mktsegment
measures:
  - name: customer_count
    expr: COUNT(DISTINCT c_custkey)
$$;

In [ ]:
%sql
ALTER VIEW workspace.metric_views_lab.orders_metric_view
ADD CONSTRAINT orders_to_customers
MANY TO ONE RELATIONSHIP TO workspace.metric_views_lab.customers_metric_view
USING (customer_key);

In [ ]:
%sql
DESCRIBE TABLE EXTENDED workspace.metric_views_lab.orders_metric_view AS JSON;

### What counts as a pass?

* The filtered measure matches the direct `FILTER` control and does not remove `O` or `P` rows.
* The offset view's prior-year value matches a hand-written `ADD_MONTHS(..., -12)` control wherever a prior month exists.
* `DESCRIBE TABLE EXTENDED` exposes the relationship metadata. Remember: the relationship is a map on the noticeboard, not a structural inspection of the building.